# Longitudinal Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Danielle Aira Savellano
- AI Acknowledgements: The following code was written with assistance from GitHub Copilot.

## Text Preprocessing

### File Downloading

Meeting minutes were downloaded from the [JOB website](https://alleghenycontroller.com/the-controller/jail-oversight-board/jail-oversight-board-internal/) as `PDF` or `DOCX` files for consistency. A CSV was created using Excel for desired filenames and links for download. This notebook contains the code used to:

- Download from `Data/Scraped/download_links.csv` (`Desired Title`, `Link`, `Year`, `Mo`) into `Data/Scraped` with the desired filename and extension
- Write `Data/Scraped/download_tracker.csv` with the same columns plus `Downloaded` status and `File Type`

### Results

A total of **156 files were downloaded**. There were 116 PDFs and 40 DOCx files. Meetings ranged from March 2012 to February 2026. Furthermore, there were 4 special meetings (additional to monthly meetings).

> Files were named `YYYY_M_X_JOB_Minutes` where `YYYY` = year, `M` = Mo, and `X` = special meeting or not. For example: `2024_2_1_JOB_Minutes` → February 2024 Special Meeting

A summary by file-type, year, and special meeting status can be found at the end of the file.

In [13]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

# Paths and Session Setup
BASE = Path("..").resolve()
CSV_IN = BASE / "Data" / "Scraped" / "download_links.csv"
OUT_DIR = BASE / "Data" / "Scraped"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT = OUT_DIR / "download_tracker.csv"
sess = requests.Session()

# Helper Functions
def suffix(url: str) -> str:
    '''Extract file suffix from URL, default to .pdf if not found.'''
    s = Path(urlparse(url).path).suffix.lower()
    return s if s else ".pdf"

def desired_path(title: str, ext: str) -> Path:
    '''Generate desired path given title and extension.'''
    ext = ext if ext.startswith(".") else f".{ext}"
    if ext == ".doc":
        ext = ".docx"  # Convert .doc to .docx
    return OUT_DIR / f"{title}{ext.lower()}"

# Prepare output fields and results list
with CSV_IN.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    cols = list(reader.fieldnames or [])
    rows_in = list(reader)

extra = ["Downloaded", "File Type"]
out_fields = cols + [c for c in extra if c not in cols]
results = []
n_skipped = 0

# Processing input CSV
for row in rows_in:
    title = (row.get("Desired Title")).strip()
    url = (row.get("Link")).strip()
    out = {**row}                                           # Keep original data
    ext = suffix(url)                                       # Get file extension

    target = desired_path(title, ext)                       # Desired path (combines title and extension)

    if target.is_file() and target.stat().st_size > 0:      # If file exists and is not empty
        n_skipped += 1
        out["Downloaded"], out["File Type"] = "True", target.suffix.lower()
        results.append(out)
        continue                                            # Skip download

    try:
        r = sess.get(url, timeout=90)
        r.raise_for_status()
        target.write_bytes(r.content)
        out["Downloaded"], out["File Type"] = "True", ext   # Add download status and file type
    except Exception as e:
        out["Downloaded"], out["File Type"] = "False", ext
        print(f"{title}: {e}")

    results.append(out)                                     # Append result for document
    time.sleep(0.2)

# Write results to output CSV
with CSV_OUT.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=out_fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(results)

# Summary
downloaded = sum(1 for r in results if r.get("Downloaded") == "True")
print(f"{downloaded}/{len(results)} Downloaded ({n_skipped} Already Existing)")
print(f"Saved to: {OUT_DIR}")
print(f"Tracker: {CSV_OUT}")

156/156 Downloaded (156 Already Existing)
Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/board_meeting_analysis/Data/Scraped
Tracker: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/board_meeting_analysis/Data/Scraped/download_tracker.csv


In [12]:
# Download Summary

print(f"\n{len(results)} Total Files")

print("\nFile Types")
for file_type in set(r.get("File Type") for r in results):
    count = sum(1 for r in results if r.get("File Type") == file_type)
    print(f"{file_type}: {count}")

print("\nYears")

for year in sorted(set(r.get("Year") for r in results)):
    count = sum(1 for r in results if r.get("Desired Title").split("_")[0] == year)
    print(f"{year}: {count}")


156 Total Files

File Types
.pdf: 116
.docx: 40

Years
2012: 7
2013: 8
2014: 10
2015: 9
2016: 11
2017: 12
2018: 12
2019: 11
2020: 10
2021: 13
2022: 12
2023: 12
2024: 14
2025: 13
2026: 2


In [11]:
special_count = 0

print("\nSpecial Meetings:")
for r in results:
    if r.get("Desired Title").split("_")[2] == "1":
        special_count += 1
        print(r.get("Desired Title"))

print(f"\nTotal Special Meetings: {special_count}")


Special Meetings:
2021_9_1_JOB_Minutes
2024_2_1_JOB_Minutes
2024_3_1_JOB_Minutes
2025_8_1_JOB_Minutes

Total Special Meetings: 4
